# Dump Opsim Database

Goal Dump opsim databases:
- https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/maf/
- rubin_sim : https://github.com/lsst/rubin_sim



- author : Sylvie Dagoret-Campagne
- creation : 2026-08-10
- last update : 2026-08-10

## 1. Import

In [ ]:
import warnings
import logging
import urllib
import os
from os import path
import numpy as np
import pandas as pd

# check which columns are available
import sqlite3

from collections import OrderedDict

import matplotlib.pyplot as plt
from scipy.constants import golden
import rubin_sim.maf.db

from rubin_sim.data import get_baseline

from astropy.time import Time

In [ ]:
%matplotlib inline
# %config InlineBackend.figure_format = 'svg'
# %load_ext lab_black
# %load_ext pycodestyle_magic
# %flake8_on --ignore E501,W505
%load_ext autoreload
%autoreload 1

In [ ]:
def mjd_to_datestr(mjd):
    """MJD (TAI) → ISO date string YYYY-MM-DD."""
    try:
        return Time(float(mjd), format="mjd", scale="tai").isot[:10]
    except Exception:
        return "?"

In [ ]:
def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 12) -> None:
    """
    Add a secondary x-axis on **top** of *ax* showing calendar dates (YYYY-MM-DD),
    inclined 40 degrees to the left for readability.
    """
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return

    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return

    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)

    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=7, labelpad=6)


print("mjd_to_datestr() and add_date_axis_on_top() defined.")

In [ ]:
params = {
    "legend.fontsize": "xx-large",
    #          'figure.figsize': (15, 5),
    "axes.labelsize": "xx-large",
    "axes.titlesize": "xx-large",
    "xtick.labelsize": "xx-large",
    "ytick.labelsize": "xx-large",
}
plt.rcParams.update(params)

In [ ]:
warnings.filterwarnings(
    "ignore",
    append=True,
    message=r".*Tried to get polar motions for times after IERS data is valid.*",
)
warnings.filterwarnings("ignore", append=True, message=r".*dubious year.*")

## 2. Logging

In [ ]:
logging.basicConfig(format="%(asctime)s %(message)s")
logger = logging.getLogger("hourglass_notebook")
logger.setLevel("DEBUG")
logger.info("Starting")

## 3. Configuration

In [ ]:
# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "01_DUMPOPSIM"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Output data: {os.path.abspath(DIR_DATA)}")
print(f"Figures    : {os.path.abspath(DIR_FIGS)}")

In [ ]:
def savefig(name: str):
    """Save figure to both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 3. Get input  database connections

In [ ]:
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"
path_topdir = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"path_topdir = {path_topdir}")

In [ ]:
logger.debug("Configuring database connections")
baseline_file = get_baseline()

In [ ]:
run_name = os.path.split(baseline_file)[-1].replace(".db", "")
print(run_name)

## 4. Dump

In [ ]:
opsim_fname = baseline_file
conn = sqlite3.connect(opsim_fname)
cursor = conn.cursor()

In [ ]:
df = pd.read_sql(
    """
    SELECT *
    FROM observations
    LIMIT 1
    """,
    conn,
)

print(df.columns.tolist())

### 4.1 Table Observation dump

In [ ]:
cursor.execute("PRAGMA table_info(observations);")
cols = cursor.fetchall()

In [ ]:
list(cols)

In [ ]:
cols_to_check = ["scheduler_note", "target_name", "observation_reason", "science_program"]

for col in cols_to_check:
    try:
        print(f"\n=== {col} ===")
        # cursor.execute(f"SELECT DISTINCT {col} FROM observations;")

        cursor.execute(
            f"""
            SELECT {col}, COUNT(*) 
            FROM observations 
            GROUP BY {col}
            ORDER BY COUNT(*) DESC
            LIMIT 50;
            """
        )

        values = cursor.fetchall()
        for v in values:
            print(v[0])
    except Exception as e:
        print(f"{col} -> erreur ({e})")

In [ ]:
conn = sqlite3.connect(baseline_file)

for col in ["scheduler_note", "observation_reason", "target_name", "science_program"]:
    print("\n======", col, "======")
    df = pd.read_sql(
        f"""
        SELECT {col}, COUNT(*) as n
        FROM observations
        GROUP BY {col}
        ORDER BY n DESC
        LIMIT 30
        """,
        conn,
    )
    print(df)

In [ ]:
df = pd.read_sql(
    """
    SELECT scheduler_note,
           observation_reason,
           target_name
    FROM observations
    LIMIT 10000
    """,
    conn,
)

df.to_records(index=False)

## 5. Plots

### 5.1 Barplots

In [ ]:
# Plot bar charts for categorical columns
import seaborn as sns
import matplotlib.ticker as ticker


# Function to group rare categories (less than threshold of total)
def group_rare_categories(df_counts, threshold=0.001):
    total = df_counts["n"].sum()
    mask = df_counts["n"] / total >= threshold
    grouped = df_counts[mask]
    if len(grouped) < len(df_counts):
        other_count = df_counts[~mask]["n"].sum()
        grouped = pd.concat([grouped, pd.DataFrame({df_counts.columns[0]: ["Other"], "n": [other_count]})])
    return grouped.sort_values("n", ascending=False)


# Set up the figure
plt.figure(figsize=(18, 18))
sns.set_style("whitegrid")

# Columns to visualize
columns_to_plot = ["scheduler_note", "observation_reason", "target_name", "science_program"]
n_cols = 2
n_rows = 2

# Get database connection
conn = sqlite3.connect(baseline_file)

# loop on columns to plto thus on figures
for idx, col in enumerate(columns_to_plot):
    # Fetch counts for this column
    df_counts = pd.read_sql(
        f"""
        SELECT {col}, COUNT(*) as n
        FROM observations
        GROUP BY {col}
        ORDER BY n DESC
        """,
        conn,
    )

    # Group rare categories if there are too many
    if len(df_counts) > 15:
        df_counts = group_rare_categories(df_counts, threshold=0.001)
    else:
        df_counts = df_counts.sort_values("n", ascending=False)

    # Create subplot
    ax = plt.subplot(n_rows, n_cols, idx + 1)

    # Horizontal bar plot with color palette
    colors = sns.color_palette("viridis_r", len(df_counts))
    bars = ax.barh(range(len(df_counts)), df_counts["n"], color=colors)

    # Customize plot
    ax.set_yticks(range(len(df_counts)))
    ax.set_yticklabels(df_counts[col].astype(str), fontsize=10)
    ax.invert_yaxis()  # Top category on top
    ax.set_xlabel("Count")
    ax.set_title(f"Distribution of {col}")
    ax.tick_params(axis="x", rotation=45)

    # Add count labels on bars
    for i, (bar, count) in enumerate(zip(bars, df_counts["n"])):
        ax.text(count, i, f" {count:,}", va="center", fontsize=12)

    # Format x-axis
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.suptitle(f"run : {run_name}", fontsize=20, fontweight="bold")
plt.tight_layout()
# plt.savefig("categorical_columns_distribution.png", dpi=150, bbox_inches="tight")
savefig("categorical_columns_distribution")
plt.show()

### 5.2 Plot vs Time

## 